Following notebook has been worked upon in collab. To reproduce the result, upload it on collab and then in files section upload the training csv and test csv

The procedure followed was the preprocessing of the complaint_text followed by the vectorization of words via Bag of words technique and labelling of primary_category, secondary_category and severity. Then I tried several ML algorithms via Scikit to see which best suits the data. I tried Naive Byes, Random Forest and Logistic Regression by first training model on 90% of the train data and validating on rest 10%. After seeing the result it was eveident that logistic regression was the best for it. So I decided to use it for training on complete data.

In [3]:
import numpy as np
import pandas as pd


In [4]:
df=pd.read_csv('/content/train_complaints.csv')

In [5]:
df.head()

,complaint_id,complaint_text,primary_category,secondary_category,severity
0,1634299,Back into XXXX of 2010 during this mortgage cr...,Mortgage,"Loan modification,collection,foreclosure",2
1,5505088,I checked my credit report and I am upset on w...,"Credit reporting, credit repair services, or o...",Problem with a credit reporting company's inve...,1
2,10979675,I am writing to dispute the accuracy of the in...,Credit reporting or other personal consumer re...,Problem with a company's investigation into an...,1
3,7520351,A transaction from XXXX XXXX XXXX submitted a ...,Checking or savings account,Managing an account,1
4,5847870,I was recently alerted to an account in collec...,Debt collection,Attempts to collect debt not owed,5


Counting total labels in each category

In [6]:
df['primary_category'].value_counts()


,count
primary_category,
Credit reporting or other personal consumer reports,735
Mortgage,611
Debt collection,579
"Credit reporting, credit repair services, or other personal consumer reports",463
Credit reporting,308
Checking or savings account,292
Credit card,5
Vehicle loan or lease,3
Credit card or prepaid card,3


In [7]:
df['secondary_category'].value_counts()

,count
secondary_category,
Problem with a company's investigation into an existing problem,311
Problem with a credit reporting company's investigation into an existing problem,310
Trouble during payment process,309
Incorrect information on credit report,308
"Loan modification,collection,foreclosure",301
Incorrect information on your report,298
Attempts to collect debt not owed,295
Managing an account,292
Improper use of your report,291


## Data preprocessing

Removal of special characters

In [8]:
import re
def remove_tags(text):
  clean_text=re.sub(re.compile('<.$%&*()>{}?/#@!~''",+-'), '', text)
  return clean_text

In [9]:
df['complaint_text']=df['complaint_text'].apply(remove_tags)

Turning all the letters in lowercase

In [10]:
df['complaint_text']=df['complaint_text'].apply(lambda x:x.lower())

In [11]:
df


,complaint_id,complaint_text,primary_category,secondary_category,severity
0,1634299,back into xxxx of 2010 during this mortgage cr...,Mortgage,"Loan modification,collection,foreclosure",2
1,5505088,i checked my credit report and i am upset on w...,"Credit reporting, credit repair services, or o...",Problem with a credit reporting company's inve...,1
2,10979675,i am writing to dispute the accuracy of the in...,Credit reporting or other personal consumer re...,Problem with a company's investigation into an...,1
3,7520351,a transaction from xxxx xxxx xxxx submitted a ...,Checking or savings account,Managing an account,1
4,5847870,i was recently alerted to an account in collec...,Debt collection,Attempts to collect debt not owed,5
...,...,...,...,...,...
2994,13965958,the servicing of my mortgage account was trans...,Mortgage,Trouble during payment process,1
2995,9622950,i am writing to delete the following informati...,Credit reporting or other personal consumer re...,Incorrect information on your report,5
2996,14268251,equifax information services xxxx xxxx xxxx xx...,Credit reporting or other personal consumer re...,Incorrect information on your report,5
2997,13284349,i have not supplied proof under the doctrine o...,Credit reporting or other personal consumer re...,Incorrect information on your report,1


Removal of stop words like is,the etc

In [12]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
sw_list=stopwords.words('english')
df['complaint_text']=df['complaint_text'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
df


,complaint_id,complaint_text,primary_category,secondary_category,severity
0,1634299,back xxxx 2010 mortgage crash wife part second...,Mortgage,"Loan modification,collection,foreclosure",2
1,5505088,checked credit report upset found out. late pa...,"Credit reporting, credit repair services, or o...",Problem with a credit reporting company's inve...,1
2,10979675,writing dispute accuracy information regarding...,Credit reporting or other personal consumer re...,Problem with a company's investigation into an...,1
3,7520351,transaction xxxx xxxx xxxx submitted withdrawa...,Checking or savings account,Managing an account,1
4,5847870,recently alerted account collections sum {$500...,Debt collection,Attempts to collect debt not owed,5
...,...,...,...,...,...
2994,13965958,servicing mortgage account transferred xxxxxxx...,Mortgage,Trouble during payment process,1
2995,9622950,writing delete following information file. ite...,Credit reporting or other personal consumer re...,Incorrect information on your report,5
2996,14268251,equifax information services xxxx xxxx xxxx xx...,Credit reporting or other personal consumer re...,Incorrect information on your report,5
2997,13284349,"supplied proof doctrine estoppel silence, enge...",Credit reporting or other personal consumer re...,Incorrect information on your report,1


In [14]:
X=df['complaint_text']
Y=df['primary_category']
Z=df['secondary_category']


In [15]:
X

,complaint_text
0,back xxxx 2010 mortgage crash wife part second...
1,checked credit report upset found out. late pa...
2,writing dispute accuracy information regarding...
3,transaction xxxx xxxx xxxx submitted withdrawa...
4,recently alerted account collections sum {$500...
...,...
2994,servicing mortgage account transferred xxxxxxx...
2995,writing delete following information file. ite...
2996,equifax information services xxxx xxxx xxxx xx...
2997,"supplied proof doctrine estoppel silence, enge..."


Encoding each label

In [18]:
from sklearn.preprocessing import LabelEncoder
primary_encoder = LabelEncoder()
secondary_encoder = LabelEncoder()
Y=primary_encoder.fit_transform(Y)


In [19]:
Y

array([7, 5, 4, ..., 4, 4, 6])

In [20]:
Z=secondary_encoder.fit_transform(Z)

In [21]:
Z

array([4, 7, 6, ..., 3, 3, 9])

# Model for Complaint_text and primary_category

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.1,random_state=1)

In [23]:
X_train.shape

(2699,)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

In [25]:
cv_primary = CountVectorizer(ngram_range=(1,2))
cv_secondary = CountVectorizer(ngram_range=(1,2))
cv_severity = CountVectorizer(ngram_range=(1,2))

In [26]:
X_train_bow = cv_primary.fit_transform(X_train).toarray()
X_test_bow = cv_primary.transform(X_test).toarray()

In [27]:
from sklearn.linear_model import LogisticRegression
lrlog1=LogisticRegression()
lrlog1.fit(X_train_bow,Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [28]:
Y_pred=lrlog1.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Y_test, Y_pred)

0.7766666666666666

In [29]:
from sklearn.naive_bayes import GaussianNB
lrNB1=GaussianNB()
lrNB1.fit(X_train_bow,Y_train)

GaussianNB()

In [30]:
Y_pred=lrNB1.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Y_test, Y_pred)

0.6666666666666666

In [31]:
from sklearn.ensemble import RandomForestClassifier
lrRF1=RandomForestClassifier()
lrRF1.fit(X_train_bow,Y_train)

RandomForestClassifier()

In [32]:
Y_pred=lrRF1.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Y_test, Y_pred)

0.7433333333333333

From above it can be concluded that logistic regression is best for this task. Hence now we train a model on complete data using logistic regression.

Final model for Complain text and Primary Category

In [33]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.001,random_state=1)

In [34]:
X_train_bow = cv_primary.fit_transform(X_train).toarray()
X_test_bow = cv_primary.transform(X_test).toarray()

In [35]:
from sklearn.linear_model import LogisticRegression
model1=LogisticRegression()
model1.fit(X_train_bow,Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

# Model for Complaint_Text and Secondary_category

In [36]:
from sklearn.model_selection import train_test_split
X_train2,X_test2,Z_train,Z_test=train_test_split(X,Z,test_size=0.1,random_state=1)

In [37]:
X_train2_bow = cv_secondary.fit_transform(X_train2).toarray()
X_test2_bow = cv_secondary.transform(X_test2).toarray()

In [38]:
from sklearn.linear_model import LogisticRegression
lrlog2=LogisticRegression(random_state=16)
lrlog2.fit(X_train2_bow,Z_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=16)

In [39]:
Z_pred=lrlog2.predict(X_test2_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Z_test, Z_pred)

0.63

In [40]:
from sklearn.naive_bayes import GaussianNB
lrNB2=GaussianNB()
lrNB2.fit(X_train2_bow,Z_train)

GaussianNB()

In [43]:
Z_pred=lrNB2.predict(X_test2_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Z_test, Z_pred)

0.59

In [44]:
from sklearn.ensemble import RandomForestClassifier
lrRF2=RandomForestClassifier()
lrRF2.fit(X_train2_bow,Z_train)

RandomForestClassifier()

In [45]:
Z_pred=lrRF2.predict(X_test2_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(Z_test, Z_pred)

0.6266666666666667

In this case again logistic regression is slightly better than Random Forest, so we will use it again to train a model on complete data


Final model for Complaint text and Secondary category

In [46]:
from sklearn.model_selection import train_test_split
X_train,X_test,Z_train,Z_test=train_test_split(X,Z,test_size=0.001,random_state=1)

In [47]:
X_train_bow = cv_secondary.fit_transform(X_train).toarray()
X_test_bow = cv_secondary.transform(X_test).toarray()

In [48]:
from sklearn.linear_model import LogisticRegression
model2=LogisticRegression()
model2.fit(X_train_bow,Z_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

# Model for Complaint Text and Severity

In [49]:
from sklearn.preprocessing import LabelEncoder
severity_encoder=LabelEncoder()
S = df['severity']
S = severity_encoder.fit_transform(S)

In [50]:
from sklearn.model_selection import train_test_split
X_train3,X_test3,S_train,S_test=train_test_split(X,S,test_size=0.1,random_state=1)

In [51]:
X_train3_bow = cv_severity.fit_transform(X_train3).toarray()
X_test3_bow = cv_severity.transform(X_test3).toarray()

In [52]:
from sklearn.linear_model import LogisticRegression
lrlog3=LogisticRegression()
lrlog3.fit(X_train3_bow,S_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [53]:
S_pred=lrlog3.predict(X_test3_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(S_test, S_pred)

0.87

In [54]:
from sklearn.naive_bayes import GaussianNB
lrNB3=GaussianNB()
lrNB3.fit(X_train3_bow,S_train)

GaussianNB()

In [56]:
Y_pred=lrNB3.predict(X_test3_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(S_test, S_pred)

0.87

In [58]:
from sklearn.ensemble import RandomForestClassifier
lrRF3=RandomForestClassifier()
lrRF3.fit(X_train3_bow,S_train)

RandomForestClassifier()

In [59]:
S_pred=lrRF3.predict(X_test3_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(S_test, S_pred)

0.8633333333333333

Here naive byes and logistic regression have same accuracy over validation data. We will go with logisctic regression again, because in many cases the classification seems to overlap for eg "Credit Card" and "Credit Card and Prepaid Card" in such cases  "Logistic Regression often outperforms Naive Bayes because it better models the decision boundary between classes and can capture complex relationships between features"
Material referred:

 https://raghda-altaei.medium.com/naive-bayes-vs-logistic-regression-a-simple-guide-to-two-popular-classifiers-91cc49322792

Final model for complaint text and severity

In [60]:
from sklearn.model_selection import train_test_split
X_train,X_test,S_train,S_test=train_test_split(X,S,test_size=0.001,random_state=1)

In [61]:
X_train_bow = cv_severity.fit_transform(X_train).toarray()
X_test_bow = cv_severity.transform(X_test).toarray()

In [62]:
from sklearn.linear_model import LogisticRegression
model3=LogisticRegression()
model3.fit(X_train_bow,S_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

# Final CSV

In [63]:
test_df = pd.read_csv('/content/test_complaints.csv')
test_df['complaint_text'] = test_df['complaint_text'].apply(remove_tags)
test_df['complaint_text'] = test_df['complaint_text'].str.lower()
test_df['complaint_text'] = test_df['complaint_text'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [64]:
X_test_primary = cv_primary.transform(test_df['complaint_text'])
X_test_secondary = cv_secondary.transform(test_df['complaint_text'])
X_test_severity = cv_severity.transform(test_df['complaint_text'])

primary_pred = model1.predict(X_test_primary)
secondary_pred = model2.predict(X_test_secondary)
severity_pred = model3.predict(X_test_severity)



In [65]:
primary_pred_labels = primary_encoder.inverse_transform(primary_pred)
secondary_pred_labels = secondary_encoder.inverse_transform(secondary_pred)
severity_pred_labels = severity_encoder.inverse_transform(severity_pred)


In [66]:
wdf=pd.read_csv('/content/test_complaints.csv')
complaint_id=wdf['complaint_id']
complaint_text=wdf['complaint_text']

final_df = pd.DataFrame({
    'complaint_id': complaint_id,
    'complaint_text': complaint_text,
    'primary_category': primary_pred_labels,
    'secondary_category': secondary_pred_labels,
    'severity': severity_pred_labels
})

final_df.to_csv('/content/final_predictions.csv', index=False)

Certain techniques might further help to increase accuracy like
using WordtoVec and ML or WordtoVec and DL for better sentimental analysis which is not possible through bag of words and ML techniques used right now. However the training process will take a lot more time in that scenario.